In [ ]:
import pandas as pd
import geopandas as gpd
import glob
from shapely.geometry import Point

tracts = (
    gpd.read_file("./tract_shapefiles/tl_2023_pa_tract/tl_2023_42_tract.shp")  # Pennsylvania = FIPS 42
      .to_crs("EPSG:4326")                                   # keep everything in WGS‑84
      [["GEOID", "geometry"]]                                # trim to what we need
)


In [ ]:
# keys → ride file *stem* (or any unique token you like)
# values → station snapshot file
RIDE_TO_STATION = {
    "april-2023":  "pogoh-station-locations-april-2023.xlsx",
    "may-2023":    "pogoh-station-locations-april-2023.xlsx",   # same network as April
    "june-2023":   "pogoh-station-locations-april-2023.xlsx",
    "july-2023":   "pogoh-station-locations-april-2023.xlsx",
    "august-2023": "pogoh-station-locations-august-2023-1.xlsx",
    "september-2023": "pogoh-station-locations-august-2023-1.xlsx",
    # …continue…
}


In [ ]:
from pathlib import Path

DATA_DIR = Path("./bikeshare_data/pittsburgh_bikeshare")

STATION_GROUPS = {
    "pogo-station-locations-may-2022.xlsx":[
        "january-2023.xlsx"
    ],
    "pogo-station-locations-february-2023.xlsx":[
        "february-2023.xlsx", "march-2023.xlsx", "april-2023.xlsx", "may-2023.xlsx"
    ],
    "pogo-station-locations-june-2023.xlsx":[
        "june-2023.xlsx"
    ],
    "pogo-station-locations-july-2023.xlsx":[
        "july-2023.xlsx"
    ],
    "pogoh-station-locations-august-2023-1.xlsx": [
        "august-2023.xlsx", "september-2023.xlsx"
    ],
    "pogo-station-locations-october-2023.csv":[
        "october-2023.xlsx", "november-2023.xlsx", "december-2023.xlsx", "january-2024.xlsx", 
        "february-2024.xlsx", "march-2024.xlsx", "april-2024.xlsx", "may-2024.xlsx", 
        "june-2024.xlsx", "july-2024.xlsx", "august-2024.xlsx", "september-2024.xlsx"
    ],
    "pogo-station-locations-october-2024.xlsx":[
        "october-2024.xlsx"
    ],
    "pogo-station-locations-november-2024.xlsx":[
        "november-2024.xlsx", "december-2024.xlsx", "january-2025.xlsx"
    ],
    "pogo-station-locations-february-2025.xlsx":[
        "february-2025.xlsx"
    ]
}


In [ ]:
#!/usr/bin/env python3
"""
pgh_bikeshare_to_tracts.py
---------------------------------
Convert Pogoh Pittsburgh bikeshare ride data into **total trips per 2020 Census tract (GEOID)**.

The script expects the folder layout:

    data_root/
        rides/               # all monthly ride workbooks (XLSX)
        stations/            # all station snapshots (xlsx / csv)
        tracts/tl_2020_42_tract.shp  # 2020 PA census tracts from TIGER

and will create

    outputs/pgh_bike_total_trips_by_geoid.csv

Adjust the three PATH constants below if your structure differs.

For each station snapshot listed in ``STATION_GROUPS`` it
  1. loads the snapshot once and builds a GeoDataFrame of dock points;
  2. loops over every ride workbook mapped to it, localising timestamps
     robustly across DST boundaries (``ambiguous=True`` solves the
     multi‑year problem);
  3. attaches start / end coordinates, spatial‑joins to the tract layer;
  4. tallies outbound, inbound and total trips per tract;
  5. finally writes one tidy CSV.

Requires: pandas ≥1.5, geopandas ≥0.14, shapely ≥2.0, pyarrow (for parquet).
"""
from __future__ import annotations

from pathlib import Path
from collections import defaultdict
import warnings

import pandas as pd
import geopandas as gpd

# ---------------------------------------------------------------------------
# Configuration -------------------------------------------------------------
# ---------------------------------------------------------------------------

# Root folders – tweak as needed
DATA_ROOT = Path("./bikeshare_data/pittsburgh_bikeshare")
STATION_DIR = DATA_ROOT  # station snapshots live in the same folder in this project
TRACT_FILE: Path = Path("./tract_shapefiles/tl_2023_pa_tract/tl_2023_42_tract.shp")
OUTPUTS = Path("./bikeshare_data/tracts_with_trips")
OUTPUTS.mkdir(parents=True, exist_ok=True)

# Time‑zone for Pittsburgh
TZ = "America/New_York"

# Manual mapping   station‑snapshot → [ride workbooks]
STATION_GROUPS: dict[str, list[str]] = {
    "pogoh-station-locations-may-2022.xlsx": [
        "january-2023.xlsx"
    ],
    "pogoh-station-locations-february-2023.xlsx": [
        "february-2023.xlsx", "march-2023.xlsx", "apr-2023.xlsx", "may-2023.xlsx"
    ],
    "pogoh-station-locations-june-2023.xlsx": [
        "june-2023.xlsx"
    ],
    "pogoh-station-locations-july-2023.xlsx": [
        "july-2023.xlsx"
    ],
    "pogoh-station-locations-august-2023-1.xlsx": [
        "august-2023.xlsx", "september-2023.xlsx",
    ],
    "pogoh-station-locations-october-2023.csv": [
        "october-2023.xlsx", "november-2023.xlsx", "december-2023.xlsx",
        "january-2024.xlsx", "february-2024.xlsx", "march-2024.xlsx", "april-2024.xlsx",
        "may-2024.xlsx", "june-2024.xlsx", "july-2024.xlsx", "august-2024.xlsx",
        "september-2024.xlsx"
    ],
    "pogoh-station-locations-october-2024.xlsx": [
        "october-2024.xlsx"
    ],
    "pogoh-station-locations-november-2024.xlsx": [
        "november-2024.xlsx", "december-2024.xlsx", "january-2025.xlsx"
    ],
    "pogoh-station-locations-february-2025.xlsx": [
        "february-2025.xlsx"
    ],
}

# ---------------------------------------------------------------------------
# Helper functions ----------------------------------------------------------
# ---------------------------------------------------------------------------

def localize(series: pd.Series) -> pd.Series:
    """Parse strings → tz‑aware datetimes (robust across many DST years).

    Using ``ambiguous=True`` chooses the *earlier* (DST) occurrence of the
    repeated fall‑back hour, avoiding the pandas ``AmbiguousTimeError`` that
    arises when the series spans multiple years and therefore multiple DST
    transitions.
    """
    return (
        pd.to_datetime(series, errors="coerce")
          .dt.tz_localize(
              TZ,
              ambiguous=True,           # <- key change! choose DST copy
              nonexistent="shift_forward",  # spring gap → first valid time
          )
    )


def load_station(fp: Path) -> gpd.GeoDataFrame:
    """Read a station snapshot (xlsx *or* csv) → GeoDataFrame indexed by Id."""
    if fp.suffix.lower() == ".csv":
        df = pd.read_csv(fp).rename(columns=str.strip)
    else:
        df = pd.read_excel(fp).rename(columns=str.strip)

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
        crs="EPSG:4326",
    ).set_index("Id")[["geometry"]]  # keep only geometry
    return gdf


# ---------------------------------------------------------------------------
# Main pipeline -------------------------------------------------------------
# ---------------------------------------------------------------------------

def main():
    # ---- 0. Read tract layer (once) --------------------------------------
    tracts = (
        gpd.read_file(TRACT_FILE)
           .to_crs("EPSG:4326")
           [["GEOID", "ALAND", "geometry"]]
    )

    # ---- 1. Cache station snapshots --------------------------------------
    station_cache: dict[str, gpd.GeoDataFrame] = {}
    for stn_name in STATION_GROUPS:
        stn_fp = STATION_DIR / stn_name
        if not stn_fp.exists():
            raise FileNotFoundError(f"Station file {stn_fp} missing")
        station_cache[stn_name] = load_station(stn_fp)

    # ---- 2. Process rides -------------------------------------------------
    all_parts: list[pd.DataFrame] = []

    for stn_name, ride_files in STATION_GROUPS.items():
        stn_gdf = station_cache[stn_name]

        for ride_fname in ride_files:
            ride_fp = DATA_ROOT / ride_fname
            if not ride_fp.exists():
                warnings.warn(f"Ride workbook {ride_fp.name} not found – skipping")
                continue

            rides = pd.read_excel(ride_fp).rename(columns=str.strip)

            # timezone localisation
            rides["Start Date"] = localize(rides["Start Date"])
            rides["End Date"]   = localize(rides["End Date"])

            # attach coordinates (start & end)
            rides = rides.merge(
                stn_gdf, left_on="Start Station Id", right_index=True, how="left"
            ).rename(columns={"geometry": "start_geom"})

            rides = rides.merge(
                stn_gdf, left_on="End Station Id", right_index=True, how="left"
            ).rename(columns={"geometry": "end_geom"})

            all_parts.append(rides)

    if not all_parts:
        raise RuntimeError("No rides were processed – check folder paths and mapping.")

    rides_df = pd.concat(all_parts, ignore_index=True, copy=False)

    # ---- 3. Spatial joins to tracts --------------------------------------
    start_gdf = gpd.GeoDataFrame(rides_df, geometry="start_geom", crs="EPSG:4326")
    end_gdf   = gpd.GeoDataFrame(rides_df, geometry="end_geom",   crs="EPSG:4326")

    start_with_tract = gpd.sjoin(
        start_gdf.dropna(subset=["start_geom"]), tracts, how="left", predicate="within"
    ).rename(columns={"GEOID": "start_tract"}).drop(columns="index_right")

    end_with_tract = gpd.sjoin(
        end_gdf.dropna(subset=["end_geom"]), tracts, how="left", predicate="within"
    ).rename(columns={"GEOID": "end_tract"}).drop(columns="index_right")

    # ---- 4. Aggregate counts --------------------------------------------
    # ---- 4. Aggregate counts ------------------------------------------------
    out_counts = start_with_tract.groupby("start_tract", observed=True).size()
    in_counts  = end_with_tract.groupby("end_tract",   observed=True).size()

    totals = (out_counts + in_counts).rename("total_trips").to_frame()

    # merge land-area from the tract table (now keyed on GEOID)
    totals = totals.join(tracts.set_index("GEOID")["ALAND"], how="left")

    # 790 days between January 1 2023 and February 28, 2025
    totals["trips_per_day"] = totals["total_trips"] / 790
    totals = totals.astype({"total_trips": "int32"})


    # ---- 5. Write -----------------------------------------------------------------
    out_fp = OUTPUTS / "pittsburgh_tracts_with_trips.csv"
    totals.to_csv(out_fp, index_label="GEOID")
    print(f"✅ Wrote {len(totals)} rows → {out_fp}")


if __name__ == "__main__":
    main()
